In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from enum import IntEnum

In [ ]:

x = np.arange(-1, 1, 0.01)
y = np.arange(-2, 2, 0.01)
t = np.arange(0, 5, 0.1)
z = np.arange(-3, 3, 1)

In [ ]:
T, X, Y, Z = np.meshgrid(t, x, y, z, indexing='ij')

In [ ]:
T.shape

In [ ]:
psi_fake = np.array([
    [[1, 2, 3], 
     [4, 5, 6]], 
     [[7, 8, 9], 
      [10, 11, 12]]
])
psi_fake

In [ ]:
np.gradient(psi_fake, axis=3)

In [ ]:
class Coordinates(IntEnum):
    T = 0
    X = 1
    Y = 2
    Z = 3

def diff(f, x, order=1):
    if order < 0:
        raise ValueError("Order must be non-negative")
    if order == 0:
        return f
    if order == 1:
        return torch.autograd.grad(f, x, grad_outputs=torch.ones_like(f), create_graph=True)[0]
    if order > 1:
        return diff(diff(f, x, order=order-1), x, order=1)

class ODESolver:
    def __init__(self, hidden_size=64, num_layers=5):
        super().__init__()

        hidden_layers = []
        for _ in range(num_layers):
            hidden_layers.append(nn.Linear(hidden_size, hidden_size))
            hidden_layers.append(nn.ReLU())
        self.model = nn.Sequential(
            nn.Linear(1, hidden_size),
            nn.ReLU(),
            *hidden_layers,
            nn.Linear(hidden_size, 1)
        )
    
    def loss(self, t, ic):
        loss_ic = nn.MSELoss()(self.model(torch.zeros_like(t)), ic)
        f = self.model(t)
        de = f + diff(f, t, order=1)
        loss_de = nn.MSELoss()(de, torch.zeros_like(de))
        return loss_ic + loss_de

    def train(self, t, ic, epochs=1000, lr=1e-3):
        optimizer = torch.optim.Adam(self.model.parameters(), lr=lr)
        loss_fn = nn.MSELoss()

        for epoch in range(epochs):
            self.model.train()
            optimizer.zero_grad()
            loss = self.loss(t, ic)
            loss.backward()
            optimizer.step()

    def predict(self, x):
        return self.model(x)

In [ ]:
input_dim = 1
num_samples = 200
# inputs = torch.randn(num_samples, input_dim, requires_grad=True)
inputs = torch.linspace(0, 1, num_samples, requires_grad=True).unsqueeze(1)  # Shape: (num_samples, 1)
model = SimpleNN(input_dim)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

def diff(y, x, coordinate: Coordinates, order=1):
    if order == 0:
        return y 
    if order < 0:
        raise ValueError("Order must be a non-negative integer.")
    if order == 1:
        return torch.autograd.grad(y, x, grad_outputs=torch.ones_like(y), create_graph=True)[0][:, coordinate]
    if order > 1:
        return diff(diff(y, x, coordinate, order=order-1), x, coordinate, order=1)

# Define a loss function that aims to satisfy a differential equation.
def loss_fn(X, Y, initial_condition_pred, initial_condition_target=2):   
    return torch.mean((Y + diff(Y, x=X, coordinate=Coordinates.T)) ** 2) + torch.mean((initial_condition_pred - initial_condition_target) ** 2)
        

def train(model, inputs, optimizer, loss_fn, epochs=100):
    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        x = inputs + torch.rand_like(inputs) * 0.01  # Add some noise to the inputs
        y = model(x)
        loss = loss_fn(x, y, model(torch.tensor([[0.0]])))
        loss.backward()
        optimizer.step()

In [ ]:
train(model, inputs, optimizer, loss_fn, epochs=4000)
model.predict(inputs)

In [ ]:
t = torch.tensor(np.arange(0, 1, 0.01), dtype=torch.float32).unsqueeze(1)
t_np = t.detach().numpy()
pred = model.predict(t).detach().numpy()
actual = 2 * np.exp(-t_np)
plt.plot(t_np, pred)
plt.plot(t_np, actual)
display(pred)

In [ ]:
# torch.autograd.grad(model(inputs), inputs, grad_outputs=torch.ones_like(y), create_graph=True, is_grads_batched=True)
display(inputs.shape)
display(model(inputs).shape)
torch.autograd.grad(model(inputs), inputs, grad_outputs=torch.ones_like(model(inputs)), create_graph=True)[0][:, Coordinates.T]

In [ ]:
model.predict(inputs)

In [ ]:
from math_utils import diff

In [ ]:
pairs = []
for _x in np.arange(-1, 1, 0.01):
    for _t in np.arange(0, 1, 0.01):
        pairs.append([_t, _x])
X = torch.tensor(pairs, dtype=torch.float32, requires_grad=True)

x = X[:,1]
t = X[:,0]
t_np = t.detach().numpy()
x_np = x.detach().numpy()

y = t**2 * x
dy_dt = diff(y, X, coordinate=0, order=1)
dy_dx = diff(y, X, coordinate=1, order=1)


In [ ]:
fig = plt.figure()
ax = fig.add_subplot(projection='3d')

ax.scatter(
    t_np, 
    x_np, 
    y.detach().numpy(), 
    label="y = x^2 * t", 
    alpha=0.5,
    c=y.detach().numpy(),
    cmap='magma'
)
plt.show()

In [ ]:
fig = plt.figure()
ax = fig.add_subplot(projection='3d')
d = dy_dt.detach().numpy() - 2*(t*x).detach().numpy()
ax.scatter(
    t_np, 
    x_np, 
    d, 
    label="Difference in theoretical and AutoGrad dy_dt",
    c=d,
    cmap='magma'
)
plt.legend()
plt.show()

fig = plt.figure()
ax = fig.add_subplot(projection='3d')

d = dy_dx.detach().numpy() - (t**2).detach().numpy()
ax.scatter(
    t_np, 
    x_np, 
    d, 
    label="Difference in theoretical and AutoGrad dy_dx",
    c=d,
    cmap='magma'
)
plt.legend()
plt.show()

In [ ]:
def laplacian(f, X):
    laplacian = torch.zeros_like(f)
    for i in range(1, X.shape[1]):
        laplacian += diff(f, X, coordinate=i, order=2)
    return laplacian

pairs = []
for t in np.arange(0, 1, 0.05):
    for x in np.arange(-1, 1, 0.05):
        for y in np.arange(-1, 1, 0.05):
            for z in np.arange(-1, 1, 0.05):
                pairs.append([t, x, y, z])
X = torch.tensor(pairs, dtype=torch.float32, requires_grad=True)
t = X[:,0]
x = X[:,1] 
y = X[:,2]
f = 3*x**2*t**2 + y**2

In [ ]:
l = laplacian(f, X)

In [ ]:
plt.scatter(t.detach().numpy(), l.detach().numpy(), label="NN Laplacian", alpha=0.5, marker='x')
plt.scatter(t.detach().numpy(), 6*t.detach().numpy()**2+2, label="Theoretical Laplacian", alpha=0.5, marker='^')
plt.xlabel("t")
plt.ylabel("Laplacian")
plt.legend()
plt.show()